# EGF Batch Processing Pipeline

**Goal:** 3D segmentation of whole cells and intracellular signal blobs, then calculate volumetric overlap between signal channels.

| Index | Channel | Segmentation |
|-------|---------|-------------|
| Ch0 | Signal blobs | Cellpose (whole-cell mask) â†’ 3D blob segmentation |
| Ch1 | Signal blobs | 3D blob segmentation |
| Ch2 | Signal blobs | 3D blob segmentation |
| Ch3 | Nucleus | 3D blob segmentation (no overlap analysis) |

**Overlaps to compute per cell:**
- Ch0 blobs âˆ© Ch1 blobs
- Ch0 blobs âˆ© Ch2 blobs
- Ch1 blobs âˆ© Ch2 blobs

In [1]:
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
from scipy import ndimage as ndi
from skimage import filters, morphology, measure, segmentation

# -----Voxel dimensions--------used throughout the notebook--------
XY_PIXEL_UM = 0.064   # µm per XY pixel
Z_STEP_UM   = 0.130   # µm per Z step

## 1. Data Directory

In [2]:
DATA_DIR = Path(r"Z:\Marta\20260218\2026-02-18-Decon")

# Collect all OME-TIFF files, sorted by name
tiff_files = sorted(DATA_DIR.glob("*.ome.tiff"))

print(f"Found {len(tiff_files)} files in {DATA_DIR}\n")
for f in tiff_files:
    print(f.name)

Found 63 files in Z:\Marta\20260218\2026-02-18-Decon

H23 607-ko EGF 30min_10_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_1_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_3_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_4_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_5_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_6_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_7_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_8_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_9_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_10_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_11_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_2_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_3_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_4_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_5_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_6_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_7_MMStack_Po

## 2. Load a Single Image for Development

Pick one file to work with interactively before running the full batch.

In [3]:
# Change the index to try a different file
test_file = tiff_files[2]
print("Loading:", test_file.name)

with tifffile.TiffFile(test_file) as tif:
    img = tif.asarray()   # shape: (C, Z, Y, X)

print(f"Shape: {img.shape}  |  dtype: {img.dtype}")
print(f"Channels: Ch0 min={img[0].min():.1f} max={img[0].max():.1f}")
print(f"          Ch1 min={img[1].min():.1f} max={img[1].max():.1f}")
print(f"          Ch2 min={img[2].min():.1f} max={img[2].max():.1f}")
print(f"          Ch3 min={img[3].min():.1f} max={img[3].max():.1f}")

# Split into named channel arrays for convenience
ch0, ch1, ch2, ch3 = img[0], img[1], img[2], img[3]

Loading: H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.ome.tiff
Shape: (4, 54, 2048, 2048)  |  dtype: float32
Channels: Ch0 min=0.0 max=14497.7
          Ch1 min=0.0 max=3196.7
          Ch2 min=0.0 max=38252.0
          Ch3 min=0.0 max=629388.6


### Napari - raw channels

In [4]:
import napari

viewer = napari.Viewer(title=test_file.name)

viewer.add_image(ch0, name="Ch0 (signal)", colormap="green",  blending="additive")
viewer.add_image(ch1, name="Ch1 (signal)", colormap="magenta", blending="additive")
viewer.add_image(ch2, name="Ch2 (signal)", colormap="cyan",   blending="additive")
viewer.add_image(ch3, name="Ch3 (nucleus)", colormap="blue",  blending="additive")

<Image layer 'Ch3 (nucleus)' at 0x2989e9cef10>

### Cellpose smoke test â€” run this first

In [5]:
from cellpose import models
import numpy as np

# Tiny synthetic image â€” 5 slices, 64x64, single channel (Z, Y, X)
test_img = np.random.randint(0, 65535, (5, 64, 64), dtype=np.uint16).astype(np.float32) / 65535.0

cp_model = models.CellposeModel(gpu=True, pretrained_model="cpsam")
masks, _, _ = cp_model.eval(test_img, z_axis=0, do_3D=False, stitch_threshold=0.5)

print("Smoke test passed â€” Cellpose is working")
print("Output mask shape:", masks.shape)

C:\Users\kari\workspace\Jupyter\jupyter_napari\Lib\site-packages\cellpose\dynamics.py:524: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),
100%|██████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 3995.53it/s]

Smoke test passed â€” Cellpose is working
Output mask shape: (5, 64, 64)


In [6]:
from cellpose import models
from scipy.ndimage import gaussian_filter
import numpy as np

# --- parameters to tune ---
CELLPOSE_MODEL   = "cpsam"  # default model in Cellpose v4
CELL_DIAMETER    = 250    # None = auto-estimate; set in pixels (XY) if auto is off
STITCH_THRESHOLD = 0.5      # links 2D masks across Z slices (0â€“1; lower = more linking)
ANISOTROPY       = 2.03      # Z_voxel_size / XY_voxel_size â€” adjust to your voxel dims
BLUR_SIGMA       = 10      # Gaussian blur applied to Ch0 before segmentation â€”
                            # smears punctate dots into a smooth cell-body signal;
                            # increase if cells still fragment, decrease if they merge
# --------------------------

def norm_float(arr):
    arr = arr.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    return arr

# Blur Ch0 to turn punctate signal into a smooth cell envelope
ch0_blurred = gaussian_filter(ch0.astype(np.float32), sigma=BLUR_SIGMA)
ch0_norm    = norm_float(ch0_blurred)

# Ch3 = nucleus â€” gives Cellpose a clean anchor for cell boundaries
ch3_norm = norm_float(ch3)

# Stack as (Z, Y, X, 2): channel 0 = cytoplasm (blurred Ch0), channel 1 = nucleus (Ch3)
cyto_nuc = np.stack([ch0_norm, ch3_norm], axis=-1)   # shape: (Z, Y, X, 2)

cp_model = models.CellposeModel(gpu=True, pretrained_model=CELLPOSE_MODEL)

cell_masks, flows, styles = cp_model.eval(
    cyto_nuc,
    diameter=CELL_DIAMETER,
    z_axis=0,
    channel_axis=3,             # last axis holds the 2 channels
    do_3D=False,
    stitch_threshold=STITCH_THRESHOLD,
    anisotropy=ANISOTROPY,
)

n_cells = cell_masks.max()
print(f"Detected {n_cells} cell(s)")
print(f"Mask shape: {cell_masks.shape}  |  dtype: {cell_masks.dtype}")

100%|██████████████████████████████████████████████████████████████████████████████████| 53/53 [00:04<00:00, 11.96it/s]


Detected 29 cell(s)
Mask shape: (54, 2048, 2048)  |  dtype: uint16


### Napari - whole-cell masks

In [7]:
viewer.add_labels(cell_masks, name="Cellpose whole-cell")

<Labels layer 'Cellpose whole-cell' at 0x298b3691b10>

## 4. Pick One Cell to Develop Blob Segmentation

Isolate a single cell by its label ID so we can tune blob parameters on a manageable volume.

In [8]:
CELL_ID = 3   # change to whichever cell label you want to inspect

single_cell_mask = (cell_masks == CELL_ID)

# find_objects needs the integer label array, returns one slice per label (0-indexed)
bbox = ndi.find_objects(cell_masks)[CELL_ID - 1]

mask_crop = single_cell_mask[bbox]
ch0_crop  = ch0[bbox]
ch1_crop  = ch1[bbox]
ch2_crop  = ch2[bbox]
ch3_crop  = ch3[bbox]

print(f"Bounding box: {bbox}")
print(f"Crop shape:   {ch0_crop.shape}")

Bounding box: (slice(0, 50, None), slice(23, 385, None), slice(1498, 1948, None))
Crop shape:   (50, 362, 450)


## 5. 3D Blob Segmentation (Ch0, Ch1, Ch2)

Approach: Gaussian smooth â†’ threshold (Li method, works well for blobs) â†’ remove small objects â†’ label connected components.  
Key parameters to tune:
- `SIGMA` â€” smoothing scale (in voxels); increase if blobs merge, decrease if they fragment  
- `MIN_BLOB_VOXELS` â€” minimum blob size to keep (filters noise spots)

In [25]:
from skimage import feature, segmentation

def segment_blobs_3d(channel_crop, cell_mask,
                     sigma_small=1.0, sigma_large=3.0,
                     dog_thresh_pct=50,
                     intensity_tight_pct=25,
                     min_voxels=50, min_distance=3,
                     shrink_radius=0):
    """
    DoG + watershed blob segmentation with percentile-based detection and
    per-blob intensity boundary tightening.

    Parameters
    ----------
    channel_crop        : ndarray (Z, Y, X)
    cell_mask           : bool ndarray, True inside the cell
    sigma_small         : float, inner Gaussian sigma in voxels
    sigma_large         : float, outer Gaussian sigma in voxels
    dog_thresh_pct      : float, percentile of positive DoG values inside the
                          cell used as detection threshold (0-100).
                          Replaces the old relative-to-max approach so both
                          bright and dim blobs are detected equally.
                          Higher = stricter = fewer blobs.
    intensity_tight_pct : float, per-blob boundary tightening (0 = off).
                          After watershed each blob is trimmed: only voxels
                          above this percentile of the blob's own smoothed
                          intensity are kept.
                          25 = keep the top 75% of each blob's intensity
                          range, giving tight signal-following boundaries.
    min_voxels          : int, discard blobs smaller than this (pre and post tighten)
    min_distance        : int, min voxel distance between blob centres
    shrink_radius       : int, optional extra erosion after tightening (0 = off)

    Returns
    -------
    labels : int ndarray, labelled blobs (0 = background)
    """
    # Normalise to 0-1
    arr = channel_crop.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

    # DoG filter
    smooth_s = filters.gaussian(arr, sigma=sigma_small)
    smooth_l = filters.gaussian(arr, sigma=sigma_large)
    dog      = smooth_s - smooth_l
    dog[~cell_mask] = 0

    # Detection threshold: percentile of positive DoG values inside the cell.
    # Using the distribution (not dog.max()) means a single ultra-bright blob
    # cannot raise the bar and hide all the dimmer ones.
    pos_vals = dog[cell_mask & (dog > 0)]
    if pos_vals.size == 0:
        return np.zeros_like(channel_crop, dtype=np.int32)
    thresh = np.percentile(pos_vals, dog_thresh_pct)
    binary = dog > thresh
    binary = morphology.remove_small_objects(binary, max_size=min_voxels)

    # Watershed seeds from local maxima
    coords = feature.peak_local_max(
        smooth_s * cell_mask,
        min_distance=min_distance,
        labels=binary.astype(np.int32),
    )
    seed_mask = np.zeros_like(dog, dtype=bool)
    if len(coords):
        seed_mask[tuple(coords.T)] = True
    seeds  = measure.label(seed_mask)
    labels = segmentation.watershed(-smooth_s, seeds, mask=binary)
    labels = measure.label(labels > 0)

    # Per-blob intensity tightening:
    # Re-fit each blob boundary to the actual signal by keeping only voxels
    # above the Nth percentile of that blob's own smoothed intensity.
    # This cuts the low-intensity halo that makes blobs look oversized.
    if intensity_tight_pct > 0 and labels.max() > 0:
        tight = np.zeros_like(labels)
        for lab in range(1, labels.max() + 1):
            blob_vox = labels == lab
            local_thresh = np.percentile(smooth_s[blob_vox], intensity_tight_pct)
            tight[blob_vox & (smooth_s >= local_thresh)] = lab
        labels = measure.label(tight > 0)
        labels = morphology.remove_small_objects(labels, max_size=min_voxels)
        labels = measure.label(labels > 0)

    # Optional extra erosion
    if shrink_radius > 0:
        selem  = morphology.ball(int(shrink_radius))
        shrunk = morphology.erosion(labels > 0, selem)
        labels = measure.label(shrunk)
        labels = morphology.remove_small_objects(labels, max_size=min_voxels)
        labels = measure.label(labels > 0)

    return labels


# Parameters to tune
SIGMA_SMALL          = 1      # voxels, inner DoG scale
SIGMA_LARGE          = 5.0    # voxels, outer DoG scale
DOG_THRESH_PCT       = 97   # percentile of +DoG inside cell; raise to detect fewer/surer blobs
INTENSITY_TIGHT_PCT  = 25     # per-blob boundary trim (0 = off); raise to shrink boundaries further
MIN_BLOB_VOXELS      = 20     # drop blobs smaller than this (voxels)
MIN_DISTANCE         = 2      # min voxels between blob centres
BLOB_SHRINK_RADIUS   = 1    # extra erosion radius in voxels (0 = off)

blobs_ch0 = segment_blobs_3d(ch0_crop, mask_crop,
                              sigma_small=SIGMA_SMALL, sigma_large=SIGMA_LARGE,
                              dog_thresh_pct=DOG_THRESH_PCT,
                              intensity_tight_pct=INTENSITY_TIGHT_PCT,
                              min_voxels=MIN_BLOB_VOXELS, min_distance=MIN_DISTANCE,
                              shrink_radius=BLOB_SHRINK_RADIUS)
blobs_ch1 = segment_blobs_3d(ch1_crop, mask_crop,
                              sigma_small=SIGMA_SMALL, sigma_large=SIGMA_LARGE,
                              dog_thresh_pct=DOG_THRESH_PCT,
                              intensity_tight_pct=INTENSITY_TIGHT_PCT,
                              min_voxels=MIN_BLOB_VOXELS, min_distance=MIN_DISTANCE,
                              shrink_radius=BLOB_SHRINK_RADIUS)
blobs_ch2 = segment_blobs_3d(ch2_crop, mask_crop,
                              sigma_small=SIGMA_SMALL, sigma_large=SIGMA_LARGE,
                              dog_thresh_pct=DOG_THRESH_PCT,
                              intensity_tight_pct=INTENSITY_TIGHT_PCT,
                              min_voxels=MIN_BLOB_VOXELS, min_distance=MIN_DISTANCE,
                              shrink_radius=BLOB_SHRINK_RADIUS)

print(f"Ch0 blobs: {blobs_ch0.max()}")
print(f"Ch1 blobs: {blobs_ch1.max()}")
print(f"Ch2 blobs: {blobs_ch2.max()}")

Ch0 blobs: 54
Ch1 blobs: 98
Ch2 blobs: 26


### Napari â€” blob segmentations on single cell

In [26]:
viewer2 = napari.Viewer(title=f"Cell {CELL_ID} â€” blob segmentation")

viewer2.add_image(ch0_crop, name="Ch0 (signal)", colormap="green",  blending="additive")
viewer2.add_image(ch1_crop, name="Ch1 (signal)", colormap="magenta", blending="additive")
viewer2.add_image(ch2_crop, name="Ch2 (signal)", colormap="cyan",   blending="additive")

viewer2.add_labels(blobs_ch0, name="Blobs Ch0", opacity=0.6)
viewer2.add_labels(blobs_ch1, name="Blobs Ch1", opacity=0.6)
viewer2.add_labels(blobs_ch2, name="Blobs Ch2", opacity=0.6)

<Labels layer 'Blobs Ch2' at 0x29bb507d350>

## 6. Measurements

For every blob in Ch0, Ch1, and Ch2 we record:

| Group | Columns |
|---|---|
| Identity | `file`, `cell_id`, `channel`, `blob_id` |
| Volume | `volume_voxels`, `volume_um3` |
| Intensity (mean + sum) | `ch0/1/2_mean_intensity`, `ch0/1/2_sum_intensity` |
| Position | `centroid_z/y/x_px` |
| Distance | `dist_to_surface_um`, `dist_to_nucleus_um` |
| Overlap (per cell) | `c0_c1_overlap_voxels`, `c0_c2_overlap_voxels`, `c1_c2_overlap_voxels` |

Rows are grouped by channel (C0 â†’ C1 â†’ C2) so the CSV can be filtered or pivoted easily.

In [27]:
import pandas as pd

# â”€â”€ Voxel metrics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
BLOB_METRICS = [
    'volume_voxels', 'volume_um3',
    'dist_to_surface_um', 'dist_to_nucleus_um',
    'ch0_mean_intensity', 'ch1_mean_intensity', 'ch2_mean_intensity',
    'ch0_sum_intensity',  'ch1_sum_intensity',  'ch2_sum_intensity',
]
STAT_FUNCS = ['mean', 'std', 'min', 'max', 'median']

# â”€â”€ Nucleus centroid â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def get_nucleus_centroid_um(ch3_crop, mask_crop,
                             voxel_size=(Z_STEP_UM, XY_PIXEL_UM, XY_PIXEL_UM),
                             sigma=2.0):
    """
    Threshold Ch3 (nucleus) inside the cell and return the centroid in Âµm.
    Returns None if the channel is empty.
    """
    vz, vxy, _ = voxel_size
    arr = ch3_crop.astype(np.float32)
    rng = arr.max() - arr.min()
    if rng == 0:
        return None
    arr = (arr - arr.min()) / rng
    smoothed = filters.gaussian(arr, sigma=sigma)
    thresh = filters.threshold_otsu(smoothed[mask_crop])
    nuc_bin = (smoothed > thresh) & mask_crop
    if nuc_bin.sum() == 0:
        return None
    cz, cy, cx = ndi.center_of_mass(nuc_bin)
    return np.array([cz * vz, cy * vxy, cx * vxy])  # Âµm


# â”€â”€ Per-cell measurement â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def measure_cell(file_name, cell_id, cell_masks,
                 ch0, ch1, ch2, ch3,
                 seg_params,
                 voxel_size=(Z_STEP_UM, XY_PIXEL_UM, XY_PIXEL_UM)):
    """
    Crop one cell, segment blobs in Ch0/1/2, return one row per blob.
    Rows are grouped C0 â†’ C1 â†’ C2.
    """
    vz, vxy, _ = voxel_size
    voxel_vol_um3 = vz * vxy * vxy

    single_cell_mask = (cell_masks == cell_id)
    bbox = ndi.find_objects(cell_masks)[cell_id - 1]

    mask_crop = single_cell_mask[bbox]
    ch0_crop  = ch0[bbox];  ch1_crop = ch1[bbox]
    ch2_crop  = ch2[bbox];  ch3_crop = ch3[bbox]

    blobs_ch0 = segment_blobs_3d(ch0_crop, mask_crop, **seg_params)
    blobs_ch1 = segment_blobs_3d(ch1_crop, mask_crop, **seg_params)
    blobs_ch2 = segment_blobs_3d(ch2_crop, mask_crop, **seg_params)

    bin0 = blobs_ch0 > 0;  bin1 = blobs_ch1 > 0;  bin2 = blobs_ch2 > 0
    c0_c1_ov = int((bin0 & bin1).sum())
    c0_c2_ov = int((bin0 & bin2).sum())
    c1_c2_ov = int((bin1 & bin2).sum())

    dist_surface = ndi.distance_transform_edt(mask_crop, sampling=(vz, vxy, vxy))
    nuc_um = get_nucleus_centroid_um(ch3_crop, mask_crop, voxel_size)

    rows = []
    for ch_name, blobs in [('C0', blobs_ch0), ('C1', blobs_ch1), ('C2', blobs_ch2)]:
        for prop in measure.regionprops(blobs):
            blob_mask = blobs == prop.label
            cz, cy, cx = prop.centroid
            ci = (int(round(cz)), int(round(cy)), int(round(cx)))
            dist_surf = float(dist_surface[ci])
            dist_nuc  = float(np.linalg.norm(
                np.array([cz*vz, cy*vxy, cx*vxy]) - nuc_um
            )) if nuc_um is not None else np.nan

            rows.append({
                'file': file_name, 'cell_id': cell_id,
                'channel': ch_name, 'blob_id': prop.label,
                'volume_voxels':       prop.area,
                'volume_um3':          round(prop.area * voxel_vol_um3, 4),
                'centroid_z_px':       round(cz, 2),
                'centroid_y_px':       round(cy, 2),
                'centroid_x_px':       round(cx, 2),
                'ch0_mean_intensity':  round(float(ch0_crop[blob_mask].mean()), 4),
                'ch1_mean_intensity':  round(float(ch1_crop[blob_mask].mean()), 4),
                'ch2_mean_intensity':  round(float(ch2_crop[blob_mask].mean()), 4),
                'ch0_sum_intensity':   round(float(ch0_crop[blob_mask].sum()),  2),
                'ch1_sum_intensity':   round(float(ch1_crop[blob_mask].sum()),  2),
                'ch2_sum_intensity':   round(float(ch2_crop[blob_mask].sum()),  2),
                'dist_to_surface_um':  round(dist_surf, 4),
                'dist_to_nucleus_um':  round(dist_nuc, 4) if not np.isnan(dist_nuc) else np.nan,
                'c0_c1_overlap_voxels': c0_c1_ov,
                'c0_c2_overlap_voxels': c0_c2_ov,
                'c1_c2_overlap_voxels': c1_c2_ov,
            })
    return rows


# â”€â”€ Summary: one row per cell â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def make_summary_df(df):
    """
    Collapse per-blob DataFrame into one row per cell.
    Columns: file, cell_id, overlap counts, n_blobs per channel,
             mean/std/min/max/median of every metric per channel.
    """
    rows = []
    for (fname, cid), cell_df in df.groupby(['file', 'cell_id'], sort=False):
        row = {'file': fname, 'cell_id': cid}
        for col in ['c0_c1_overlap_voxels', 'c0_c2_overlap_voxels', 'c1_c2_overlap_voxels']:
            row[col] = int(cell_df[col].iloc[0])
        for ch in ['C0', 'C1', 'C2']:
            pfx   = ch.lower()
            ch_df = cell_df[cell_df['channel'] == ch]
            row[f'{pfx}_n_blobs'] = len(ch_df)
            for metric in BLOB_METRICS:
                vals = ch_df[metric].dropna()
                if vals.empty:
                    for s in STAT_FUNCS:
                        row[f'{pfx}_{metric}_{s}'] = np.nan
                else:
                    row[f'{pfx}_{metric}_mean']   = round(float(vals.mean()),   4)
                    row[f'{pfx}_{metric}_std']    = round(float(vals.std()),    4)
                    row[f'{pfx}_{metric}_min']    = round(float(vals.min()),    4)
                    row[f'{pfx}_{metric}_max']    = round(float(vals.max()),    4)
                    row[f'{pfx}_{metric}_median'] = round(float(vals.median()), 4)
        rows.append(row)
    return pd.DataFrame(rows)


# â”€â”€ CSV writer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def save_csvs(df_blobs, out_dir, stem):
    """Write granular (per-blob) and summary (per-cell) CSVs."""
    df_blobs = df_blobs.copy()
    df_blobs['channel'] = pd.Categorical(df_blobs['channel'], ['C0', 'C1', 'C2'])
    df_blobs = df_blobs.sort_values(['file', 'cell_id', 'channel', 'blob_id']).reset_index(drop=True)

    p_blobs   = out_dir / f"{stem}_blobs.csv"
    p_summary = out_dir / f"{stem}_cell_summary.csv"

    df_blobs.to_csv(p_blobs, index=False)
    df_summary = make_summary_df(df_blobs)
    df_summary.to_csv(p_summary, index=False)

    print(f"Granular CSV  ({len(df_blobs):,} rows)                       â†’ {p_blobs.name}")
    print(f"Summary CSV   ({len(df_summary):,} rows, {len(df_summary.columns)} cols) â†’ {p_summary.name}")
    return df_blobs, df_summary


print("measure_cell(), make_summary_df(), save_csvs() defined")

measure_cell(), make_summary_df(), save_csvs() defined


### 6a. Test measurements on the single loaded image

In [28]:
seg_params = dict(
    sigma_small         = SIGMA_SMALL,
    sigma_large         = SIGMA_LARGE,
    dog_thresh_pct      = DOG_THRESH_PCT,
    intensity_tight_pct = INTENSITY_TIGHT_PCT,
    min_voxels          = MIN_BLOB_VOXELS,
    min_distance        = MIN_DISTANCE,
    shrink_radius       = BLOB_SHRINK_RADIUS,
)

all_rows = []
n_cells = int(cell_masks.max())

for cid in range(1, n_cells + 1):
    print(f"  measuring cell {cid}/{n_cells} ...", end=" ", flush=True)
    rows = measure_cell(
        file_name  = test_file.name,
        cell_id    = cid,
        cell_masks = cell_masks,
        ch0=ch0, ch1=ch1, ch2=ch2, ch3=ch3,
        seg_params = seg_params,
    )
    all_rows.extend(rows)
    print(f"{len(rows)} blobs")

df_single = pd.DataFrame(all_rows)

# Sort so C0 rows come first, then C1, then C2
df_single['channel'] = pd.Categorical(df_single['channel'], ['C0', 'C1', 'C2'])
df_single = df_single.sort_values(['cell_id', 'channel', 'blob_id']).reset_index(drop=True)

print(f"\nTotal blobs: {len(df_single)}")
print(df_single.groupby('channel')[['volume_um3','dist_to_surface_um','dist_to_nucleus_um']].describe().round(3))

  measuring cell 1/29 ... 43 blobs
  measuring cell 2/29 ... 245 blobs
  measuring cell 3/29 ... 178 blobs
  measuring cell 4/29 ... 168 blobs
  measuring cell 5/29 ... 138 blobs
  measuring cell 6/29 ... 148 blobs
  measuring cell 7/29 ... 209 blobs
  measuring cell 8/29 ... 130 blobs
  measuring cell 9/29 ... 166 blobs
  measuring cell 10/29 ... 150 blobs
  measuring cell 11/29 ... 240 blobs
  measuring cell 12/29 ... 140 blobs
  measuring cell 13/29 ... 125 blobs
  measuring cell 14/29 ... 228 blobs
  measuring cell 15/29 ... 276 blobs
  measuring cell 16/29 ... 110 blobs
  measuring cell 17/29 ... 254 blobs
  measuring cell 18/29 ... 164 blobs
  measuring cell 19/29 ... 153 blobs
  measuring cell 20/29 ... 254 blobs
  measuring cell 21/29 ... 154 blobs
  measuring cell 22/29 ... 19 blobs
  measuring cell 23/29 ... 16 blobs
  measuring cell 24/29 ... 133 blobs
  measuring cell 25/29 ... 128 blobs
  measuring cell 26/29 ... 70 blobs
  measuring cell 27/29 ... 0 blobs
  measuring cell

In [29]:
# Save both CSVs for the single test image
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

df_single, df_single_summary = save_csvs(
    pd.DataFrame(all_rows), out_dir=RESULTS_DIR, stem=test_file.stem
)
df_single_summary

Granular CSV  (4,039 rows)                       â†’ H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.ome_blobs.csv
Summary CSV   (26 rows, 158 cols) â†’ H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.ome_cell_summary.csv


,file,cell_id,c0_c1_overlap_voxels,c0_c2_overlap_voxels,c1_c2_overlap_voxels,c0_n_blobs,c0_volume_voxels_mean,c0_volume_voxels_std,c0_volume_voxels_min,c0_volume_voxels_max,...,c2_ch1_sum_intensity_mean,c2_ch1_sum_intensity_std,c2_ch1_sum_intensity_min,c2_ch1_sum_intensity_max,c2_ch1_sum_intensity_median,c2_ch2_sum_intensity_mean,c2_ch2_sum_intensity_std,c2_ch2_sum_intensity_min,c2_ch2_sum_intensity_max,c2_ch2_sum_intensity_median
0,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,1,39,0,0,13,58.2308,32.2210,26.0,135.0,...,103.1575,91.8138,46.22,238.77,63.820,1.741764e+05,8.789547e+04,93171.58,280942.19,161296.010
1,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,2,1051,26,278,80,58.1000,43.8535,21.0,291.0,...,462.7735,674.9014,45.56,3726.12,182.245,4.986115e+05,5.100009e+05,100132.52,2463649.75,300041.940
2,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,3,1053,75,665,54,82.8704,62.7445,21.0,351.0,...,772.5854,1178.2555,50.49,5273.91,358.770,3.251516e+05,4.555049e+05,35015.97,1717593.00,128400.130
3,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,4,1050,80,709,51,88.2157,90.2262,21.0,424.0,...,5926.0752,15560.7184,15.21,77019.46,318.600,7.183071e+05,1.141036e+06,125823.73,6410528.00,423364.660
4,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,5,840,27,418,49,71.6531,49.9356,22.0,226.0,...,1226.1531,2416.5036,42.33,11623.43,270.515,2.444746e+05,2.571815e+05,48266.70,970730.19,138432.445
5,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,6,180,16,803,56,68.0714,46.6265,21.0,224.0,...,1386.0310,3898.7161,63.17,18076.98,256.900,4.177424e+05,6.467596e+05,52417.76,2993087.00,194637.890
6,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,7,1523,38,822,70,95.5714,93.1048,21.0,462.0,...,4601.3354,13682.2749,50.96,77623.11,239.020,4.251411e+05,5.421586e+05,63994.22,3266569.75,240527.655
7,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,8,1149,27,246,48,83.7292,84.5115,21.0,437.0,...,9208.1764,11601.3445,90.81,38399.08,4075.000,1.417369e+06,2.100644e+06,45922.22,5857972.00,207492.280
8,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,9,744,76,418,54,71.7407,44.9130,21.0,204.0,...,583.1753,1100.5324,29.49,5347.71,150.550,1.611099e+05,1.606574e+05,42740.35,859533.12,103748.665
9,H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.o...,10,924,47,446,54,61.2593,47.6204,21.0,247.0,...,1952.3296,3685.5433,34.49,14555.81,406.830,4.153198e+05,4.935412e+05,73526.89,1673711.00,195370.310


## 7. Batch Processing

Runs the full pipeline (load â†’ whole-cell segmentation â†’ blob segmentation â†’ measure) on every TIFF in `DATA_DIR`. Results for all files are combined into one CSV.

In [30]:
# â”€â”€ Run this cell ONCE to start a fresh batch collection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Re-run it only if you want to discard previous results and start over.
all_rows_batch = []
print("all_rows_batch reset â€” ready for batch run")

all_rows_batch reset â€” ready for batch run


In [31]:
from scipy.ndimage import gaussian_filter

RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
SAVE_MASKS  = True    # save cell + blob masks as TIFFs next to each source file
SHOW_NAPARI = False   # set True to open Napari after each file (single-image mode only)

# Only match original OME-TIFFs â€” excludes saved mask TIFFs (_cell_masks, _blobs_*)
tif_files = sorted(DATA_DIR.glob("*.ome.tiff"))
print(f"Found {len(tif_files)} file(s) in {DATA_DIR}\n")

def norm_float(arr):
    arr = arr.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    return arr

for file_idx, file_path in enumerate(tif_files, start=1):
    print(f"[{file_idx}/{len(tif_files)}] {file_path.name}")
    try:
        # â”€â”€ Load â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        img   = tifffile.imread(file_path)
        ch0_b = img[0];  ch1_b = img[1]
        ch2_b = img[2];  ch3_b = img[3]

        # â”€â”€ Whole-cell segmentation (Cellpose) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        ch0_blurred = gaussian_filter(ch0_b.astype(np.float32), sigma=BLUR_SIGMA)
        cyto_nuc_b  = np.stack([norm_float(ch0_blurred), norm_float(ch3_b)], axis=-1)

        masks_b, _, _ = cp_model.eval(
            cyto_nuc_b,
            diameter         = CELL_DIAMETER,
            z_axis           = 0,
            channel_axis     = 3,
            do_3D            = False,
            stitch_threshold = STITCH_THRESHOLD,
            anisotropy       = ANISOTROPY,
        )
        n_cells_b = int(masks_b.max())
        print(f"   {n_cells_b} cell(s) detected")

        blob_vol_ch0 = np.zeros_like(masks_b, dtype=np.int32)
        blob_vol_ch1 = np.zeros_like(masks_b, dtype=np.int32)
        blob_vol_ch2 = np.zeros_like(masks_b, dtype=np.int32)
        label_offset = 0

        # â”€â”€ Measure each cell â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        for cid in range(1, n_cells_b + 1):
            print(f"   cell {cid}/{n_cells_b} ...", end=" ", flush=True)
            try:
                single_mask = (masks_b == cid)
                bbox        = ndi.find_objects(masks_b)[cid - 1]
                mask_crop   = single_mask[bbox]

                bl0 = segment_blobs_3d(ch0_b[bbox], mask_crop, **seg_params)
                bl1 = segment_blobs_3d(ch1_b[bbox], mask_crop, **seg_params)
                bl2 = segment_blobs_3d(ch2_b[bbox], mask_crop, **seg_params)

                for vol, bl in [(blob_vol_ch0, bl0),
                                (blob_vol_ch1, bl1),
                                (blob_vol_ch2, bl2)]:
                    shifted   = np.where(bl > 0, bl + label_offset, 0)
                    vol[bbox] += shifted
                label_offset += max(bl0.max(), bl1.max(), bl2.max())

                rows = measure_cell(
                    file_name  = file_path.name,
                    cell_id    = cid,
                    cell_masks = masks_b,
                    ch0=ch0_b, ch1=ch1_b, ch2=ch2_b, ch3=ch3_b,
                    seg_params = seg_params,
                )
                all_rows_batch.extend(rows)
                print(f"{len(rows)} blobs  (running total: {len(all_rows_batch)})")

            except Exception as e:
                print(f"SKIPPED cell {cid} â€” {e}")

        # â”€â”€ Save segmentation masks â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        if SAVE_MASKS:
            stem = file_path.stem
            tifffile.imwrite(DATA_DIR / f"{stem}_cell_masks.tif",  masks_b.astype(np.uint16))
            tifffile.imwrite(DATA_DIR / f"{stem}_blobs_ch0.tif",   blob_vol_ch0.astype(np.uint16))
            tifffile.imwrite(DATA_DIR / f"{stem}_blobs_ch1.tif",   blob_vol_ch1.astype(np.uint16))
            tifffile.imwrite(DATA_DIR / f"{stem}_blobs_ch2.tif",   blob_vol_ch2.astype(np.uint16))
            print(f"   masks saved")

        if SHOW_NAPARI:
            import napari
            v = napari.Viewer(title=file_path.name)
            v.add_image(ch0_b, name="Ch0", colormap="green",   blending="additive")
            v.add_image(ch1_b, name="Ch1", colormap="cyan",    blending="additive")
            v.add_image(ch2_b, name="Ch2", colormap="magenta", blending="additive")
            v.add_image(ch3_b, name="Ch3 nucleus", colormap="blue", blending="additive")
            v.add_labels(masks_b,      name="whole-cell masks")
            v.add_labels(blob_vol_ch0, name="blobs Ch0")
            v.add_labels(blob_vol_ch1, name="blobs Ch1")
            v.add_labels(blob_vol_ch2, name="blobs Ch2")

        # â”€â”€ Per-image CSVs (blobs + cell summary for this file only) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        file_rows = [r for r in all_rows_batch if r['file'] == file_path.name]
        if file_rows:
            save_csvs(pd.DataFrame(file_rows), out_dir=RESULTS_DIR, stem=file_path.stem)
            print(f"   per-image CSVs saved")

    except Exception as e:
        print(f"   FILE SKIPPED â€” {e}")

    # â”€â”€ Cumulative batch CSVs (all files so far, updated after every file) â”€â”€â”€â”€
    if all_rows_batch:
        save_csvs(pd.DataFrame(all_rows_batch), out_dir=RESULTS_DIR, stem="EGF_batch")
        print(f"   batch CSVs updated ({len(all_rows_batch):,} rows so far)")

    print()

print(f"\nBatch complete â€” {len(all_rows_batch):,} total blob measurements across "
      f"{pd.DataFrame(all_rows_batch)['file'].nunique()} file(s)")

Found 63 file(s) in Z:\Marta\20260218\2026-02-18-Decon

[1/63] H23 607-ko EGF 30min_10_MMStack_Pos0.ome_cmle.ome.tiff


100%|██████████████████████████████████████████████████████████████████████████████████| 53/53 [00:04<00:00, 11.36it/s]


   17 cell(s) detected
   cell 1/17 ... 140 blobs  (running total: 140)
   cell 2/17 ... 233 blobs  (running total: 373)
   cell 3/17 ... 164 blobs  (running total: 537)
   cell 4/17 ... 188 blobs  (running total: 725)
   cell 5/17 ... 94 blobs  (running total: 819)
   cell 6/17 ... 143 blobs  (running total: 962)
   cell 7/17 ... 187 blobs  (running total: 1149)
   cell 8/17 ... 194 blobs  (running total: 1343)
   cell 9/17 ... 218 blobs  (running total: 1561)
   cell 10/17 ... 195 blobs  (running total: 1756)
   cell 11/17 ... 104 blobs  (running total: 1860)
   cell 12/17 ... 110 blobs  (running total: 1970)
   cell 13/17 ... 9 blobs  (running total: 1979)
   cell 14/17 ... 0 blobs  (running total: 1979)
   cell 15/17 ... 0 blobs  (running total: 1979)
   cell 16/17 ... 0 blobs  (running total: 1979)
   cell 17/17 ... 0 blobs  (running total: 1979)
   masks saved
Granular CSV  (1,979 rows)                       â†’ H23 607-ko EGF 30min_10_MMStack_Pos0.ome_cmle.ome_blobs.csv
Summary 

100%|██████████████████████████████████████████████████████████████████████████████████| 53/53 [00:04<00:00, 12.14it/s]


   14 cell(s) detected
   cell 1/14 ... 222 blobs  (running total: 2201)
   cell 2/14 ... 214 blobs  (running total: 2415)
   cell 3/14 ... 137 blobs  (running total: 2552)
   cell 4/14 ... 677 blobs  (running total: 3229)
   cell 5/14 ... 165 blobs  (running total: 3394)
   cell 6/14 ... 299 blobs  (running total: 3693)
   cell 7/14 ... 263 blobs  (running total: 3956)
   cell 8/14 ... 258 blobs  (running total: 4214)
   cell 9/14 ... 75 blobs  (running total: 4289)
   cell 10/14 ... 231 blobs  (running total: 4520)
   cell 11/14 ... 124 blobs  (running total: 4644)
   cell 12/14 ... 266 blobs  (running total: 4910)
   cell 13/14 ... 0 blobs  (running total: 4910)
   cell 14/14 ... 0 blobs  (running total: 4910)
   masks saved
Granular CSV  (2,931 rows)                       â†’ H23 607-ko EGF 30min_1_MMStack_Pos0.ome_cmle.ome_blobs.csv
Summary CSV   (12 rows, 158 cols) â†’ H23 607-ko EGF 30min_1_MMStack_Pos0.ome_cmle.ome_cell_summary.csv
   per-image CSVs saved
Granular CSV  (4,910 r

...

   36 cell(s) detected
   cell 1/36 ... 174 blobs  (running total: 217841)
   cell 2/36 ... 43 blobs  (running total: 217884)
   cell 3/36 ... 259 blobs  (running total: 218143)
   cell 4/36 ... 0 blobs  (running total: 218143)
   cell 5/36 ... 207 blobs  (running total: 218350)
   cell 6/36 ... 364 blobs  (running total: 218714)
   cell 7/36 ... 143 blobs  (running total: 218857)
   cell 8/36 ... 343 blobs  (running total: 219200)
   cell 9/36 ... 267 blobs  (running total: 219467)
   cell 10/36 ... 317 blobs  (running total: 219784)
   cell 11/36 ... 295 blobs  (running total: 220079)
   cell 12/36 ... 288 blobs  (running total: 220367)
   cell 13/36 ... 286 blobs  (running total: 220653)
   cell 14/36 ... 65 blobs  (running total: 220718)
   cell 15/36 ... 54 blobs  (running total: 220772)
   cell 16/36 ... 188 blobs  (running total: 220960)
   cell 17/36 ... 193 blobs  (running total: 221153)
   cell 18/36 ... 92 blobs  (running total: 221245)
   cell 19/36 ... 120 blobs  (running 

100%|██████████████████████████████████████████████████████████████████████████████████| 38/38 [00:03<00:00, 11.42it/s]


   19 cell(s) detected
   cell 1/19 ... 90 blobs  (running total: 221965)
   cell 2/19 ... 174 blobs  (running total: 222139)
   cell 3/19 ... 206 blobs  (running total: 222345)
   cell 4/19 ... 97 blobs  (running total: 222442)
   cell 5/19 ... 114 blobs  (running total: 222556)
   cell 6/19 ... 0 blobs  (running total: 222556)
   cell 7/19 ... 150 blobs  (running total: 222706)
   cell 8/19 ... 81 blobs  (running total: 222787)
   cell 9/19 ... 0 blobs  (running total: 222787)
   cell 10/19 ... 96 blobs  (running total: 222883)
   cell 11/19 ... 71 blobs  (running total: 222954)
   cell 12/19 ... 6 blobs  (running total: 222960)
   cell 13/19 ... 11 blobs  (running total: 222971)
   cell 14/19 ... 59 blobs  (running total: 223030)
   cell 15/19 ... 

C:\Users\kari\workspace\Jupyter\jupyter_napari\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?
  return func(*args, **kwargs)


1 blobs  (running total: 223031)
   cell 16/19 ... 0 blobs  (running total: 223031)
   cell 17/19 ... 43 blobs  (running total: 223074)
   cell 18/19 ... 9 blobs  (running total: 223083)
   cell 19/19 ... 0 blobs  (running total: 223083)
   masks saved
Granular CSV  (1,208 rows)                       â†’ siNeg-1 KRASgfp EEA1_2_MMStack_Pos0.ome_cmle.ome_blobs.csv
Summary CSV   (15 rows, 158 cols) â†’ siNeg-1 KRASgfp EEA1_2_MMStack_Pos0.ome_cmle.ome_cell_summary.csv
   per-image CSVs saved
Granular CSV  (223,083 rows)                       â†’ EGF_batch_blobs.csv
Summary CSV   (1,499 rows, 158 cols) â†’ EGF_batch_cell_summary.csv
   batch CSVs updated (223,083 rows so far)


Batch complete â€” 223,083 total blob measurements across 62 file(s)
